In [ ]:
import polars as pl

# Carga del dataset
df = pl.read_parquet("data/raw/mb51_synthetic_v1.parquet")
print(f"Shape: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Columnas: {df.columns}")

Shape: (15550868, 16)
Columnas: ['Werks', 'Lgort', 'Matnr', 'Maktx', 'Bwart', 'Mjahr', 'Budat', 'Cpudt', 'Cputm', 'Menge', 'Meins', 'Mblnr', 'Zeile', 'Lifnr', 'Kunnr', 'Xblnr']
Periodo: 2025-03-26 → 2027-03-16


In [3]:
# Distribución de movimientos por Bwart
bwart_counts = (
    df.group_by("Bwart")
    .agg(pl.len().alias("n"))
    .sort("n", descending=True)
    .with_columns(
        (pl.col("n") / pl.col("n").sum() * 100).round(2).alias("pct")
    )
)
print(bwart_counts)

shape: (10, 3)
┌───────┬─────────┬───────┐
│ Bwart ┆ n       ┆ pct   │
│ ---   ┆ ---     ┆ ---   │
│ str   ┆ u32     ┆ f64   │
╞═══════╪═════════╪═══════╡
│ 601   ┆ 3369820 ┆ 21.67 │
│ 602   ┆ 3302077 ┆ 21.23 │
│ 501   ┆ 2757642 ┆ 17.73 │
│ 311   ┆ 1836994 ┆ 11.81 │
│ 502   ┆ 1530104 ┆ 9.84  │
│ 411   ┆ 1225559 ┆ 7.88  │
│ 101   ┆ 611834  ┆ 3.93  │
│ 261   ┆ 458571  ┆ 2.95  │
│ 102   ┆ 305058  ┆ 1.96  │
│ 309   ┆ 153209  ┆ 0.99  │
└───────┴─────────┴───────┘


In [4]:
# Movimientos por planta
plant_counts = (
    df.group_by("Werks")
    .agg(pl.len().alias("n"))
    .sort("n", descending=True)
)
print(plant_counts)

shape: (14, 2)
┌───────────┬─────────┐
│ Werks     ┆ n       │
│ ---       ┆ ---     │
│ str       ┆ u32     │
╞═══════════╪═════════╡
│ PLNT_MX02 ┆ 1410486 │
│ PLNT_MX06 ┆ 1402578 │
│ PLNT_US02 ┆ 1360439 │
│ PLNT_US06 ┆ 1353176 │
│ PLNT_NI02 ┆ 1270041 │
│ …         ┆ …       │
│ PLNT_US03 ┆ 905295  │
│ PLNT_MX04 ┆ 901606  │
│ PLNT_US01 ┆ 853749  │
│ PLNT_MX03 ┆ 829363  │
│ PLNT_MX05 ┆ 760289  │
└───────────┴─────────┘


In [5]:
# Volumen mensual de movimientos
monthly = (
    df.with_columns(
        pl.col("Budat").dt.strftime("%Y-%m").alias("mes")
    )
    .group_by("mes")
    .agg(pl.len().alias("n"))
    .sort("mes")
)
print(monthly)

shape: (25, 2)
┌─────────┬────────┐
│ mes     ┆ n      │
│ ---     ┆ ---    │
│ str     ┆ u32    │
╞═════════╪════════╡
│ 2025-03 ┆ 125926 │
│ 2025-04 ┆ 782649 │
│ 2025-05 ┆ 871004 │
│ 2025-06 ┆ 845530 │
│ 2025-07 ┆ 917453 │
│ …       ┆ …      │
│ 2026-11 ┆ 9270   │
│ 2026-12 ┆ 1587   │
│ 2027-01 ┆ 375    │
│ 2027-02 ┆ 84     │
│ 2027-03 ┆ 28     │
└─────────┴────────┘


In [6]:
# Distribución del ciclo 601→602
# Aproximación: tomamos una muestra de 601 y calculamos ciclos
# desde la fecha de salida hasta la fecha de retorno más cercana del mismo Matnr+Werks

# Estadísticas del ciclo desde los datos del generador (muestra 10k filas)
import math

import numpy as np

# Reconstruir ciclos desde los parámetros del generador para verificar
from rpi.config import GeneratorConfig

cfg = GeneratorConfig()
rng = np.random.default_rng(cfg.random_seed)

mu = math.log(cfg.cycle.mean_days) - 0.5 * cfg.cycle.sigma ** 2
ciclos_sample = rng.lognormal(mean=mu, sigma=cfg.cycle.sigma, size=10_000)
ciclos_sample = np.clip(ciclos_sample, 1, cfg.cycle.cap_days)

print("Ciclo 601→602 — distribución log-normal (n=10,000 simulados):")
print(f"  Media:    {ciclos_sample.mean():.1f} días")
print(f"  Mediana:  {np.median(ciclos_sample):.1f} días")
print(f"  P90:      {np.percentile(ciclos_sample, 90):.1f} días")
print(f"  P99:      {np.percentile(ciclos_sample, 99):.1f} días")
print(f"  Máximo:   {ciclos_sample.max():.1f} días")

Ciclo 601→602 — distribución log-normal (n=10,000 simulados):
  Media:    24.9 días
  Mediana:  20.7 días
  P90:      45.0 días
  P99:      85.8 días
  Máximo:   180.0 días


In [7]:
# Merma por planta
merma_por_planta = (
    df.filter(pl.col("Bwart") == "601")
    .group_by("Werks")
    .agg(
        pl.len().alias("total_601"),
        (pl.col("Xblnr") == "MERMA-NO-RETORNO").sum().alias("merma"),
    )
    .with_columns(
        (pl.col("merma") / pl.col("total_601") * 100).round(2).alias("tasa_merma_pct")
    )
    .sort("Werks")
)
print(merma_por_planta)

shape: (14, 4)
┌───────────┬───────────┬───────┬────────────────┐
│ Werks     ┆ total_601 ┆ merma ┆ tasa_merma_pct │
│ ---       ┆ ---       ┆ ---   ┆ ---            │
│ str       ┆ u32       ┆ u32   ┆ f64            │
╞═══════════╪═══════════╪═══════╪════════════════╡
│ PLNT_MX01 ┆ 223920    ┆ 4562  ┆ 2.04           │
│ PLNT_MX02 ┆ 305765    ┆ 6135  ┆ 2.01           │
│ PLNT_MX03 ┆ 179830    ┆ 3490  ┆ 1.94           │
│ PLNT_MX04 ┆ 195084    ┆ 3964  ┆ 2.03           │
│ PLNT_MX05 ┆ 164552    ┆ 3293  ┆ 2.0            │
│ …         ┆ …         ┆ …     ┆ …              │
│ PLNT_US02 ┆ 295315    ┆ 5992  ┆ 2.03           │
│ PLNT_US03 ┆ 196756    ┆ 3944  ┆ 2.0            │
│ PLNT_US04 ┆ 264081    ┆ 5254  ┆ 1.99           │
│ PLNT_US05 ┆ 247509    ┆ 4954  ┆ 2.0            │
│ PLNT_US06 ┆ 293296    ┆ 5861  ┆ 2.0            │
└───────────┴───────────┴───────┴────────────────┘


In [8]:
# Distribución del lag entre Budat y Cpudt
lag = (
    df.with_columns(
        (pl.col("Cpudt") - pl.col("Budat")).dt.total_days().alias("lag_dias")
    )
    .group_by("lag_dias")
    .agg(pl.len().alias("n"))
    .sort("lag_dias")
    .with_columns(
        (pl.col("n") / pl.col("n").sum() * 100).round(2).alias("pct")
    )
)
print(lag.head(10))

shape: (10, 3)
┌──────────┬──────────┬──────┐
│ lag_dias ┆ n        ┆ pct  │
│ ---      ┆ ---      ┆ ---  │
│ i64      ┆ u32      ┆ f64  │
╞══════════╪══════════╪══════╡
│ 0        ┆ 14571494 ┆ 93.7 │
│ 1        ┆ 366760   ┆ 2.36 │
│ 2        ┆ 367436   ┆ 2.36 │
│ 3        ┆ 34856    ┆ 0.22 │
│ 4        ┆ 35063    ┆ 0.23 │
│ 5        ┆ 35198    ┆ 0.23 │
│ 6        ┆ 34916    ┆ 0.22 │
│ 7        ┆ 34920    ┆ 0.22 │
│ 8        ┆ 35237    ┆ 0.23 │
│ 9        ┆ 34988    ┆ 0.22 │
└──────────┴──────────┴──────┘
